# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

Notes:
* This file must be in the same folder as "utils.py"

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

In [2]:
# Directory paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 5): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")
# serotype = input("Serotype (e.g. H5N1): ")
# serotypes = list(serotype)
# genotypes = input("Genotypes (separate with commas and no spaces in between genotypes): ")
# genotypes = genotypes.split(",")

# Dates and locations
locations = "Antarctica,North America,South America"
start_date = "11-01-2021"
end_date = "02-20-2026"
prev_end_date = "01-23-2026"
date_range = start_date + "--" + end_date
prev_date_range = start_date + "--" + prev_end_date

# Maintenance serotypes and genotypes
serotypes = ["H5N1"]
genotypes = ["B3.13", "D1.1", "D1.3"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"

downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 
prev_downloads_saved = home + "NCBI_Virus/downloads/" + prev_date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 

andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# All serotypes and genotypes
# serotype = ""
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])


## Downloading Data

In [3]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads_saved):
    if len(files) != 0:
        break 
    else: # If we don't have any downloaded files
        # Get files
        open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

## De-Duplication

In [4]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Isolate, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Isolate")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments

143517


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PZ016555.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Canidae,brain,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8
1,PZ016556.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Canidae,brain,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8
2,PZ016557.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Canidae,brain,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8
3,PZ016558.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Canidae,brain,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8
4,PZ016559.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Canidae,brain,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135476,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
135477,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
135478,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
135479,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8


## Add sequences to dataframe

In [5]:
# NCBI Virus Naming Convention:
# "Accession|GenBank_Title|Assembly|SRA Accession|BioSample|BioProject|Genotype|Isolate|Geo Location|Host|Collection Date"

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(sequences_fasta["full_header"])

# Extract segment number so that we can add the correct sequences to the correct sample
# sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
# print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on="Accession") # , "Segment"])

0         >PZ016555.1 |Influenza A virus (A/Fox/WA/W2602...
1         >PZ016556.1 |Influenza A virus (A/Fox/WA/W2602...
2         >PZ016557.1 |Influenza A virus (A/Fox/WA/W2602...
3         >PZ016558.1 |Influenza A virus (A/Fox/WA/W2602...
4         >PZ016559.1 |Influenza A virus (A/Fox/WA/W2602...
                                ...                        
143512    >OK205883.1 |Influenza A virus (A/chicken/Vera...
143513    >OK205884.1 |Influenza A virus (A/chicken/Vera...
143514    >OK205885.1 |Influenza A virus (A/chicken/Vera...
143515    >OK205886.1 |Influenza A virus (A/chicken/Vera...
143516    >OK205887.1 |Influenza A virus (A/chicken/Vera...
Name: full_header, Length: 143517, dtype: object
143517
124080


In [6]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PZ016555.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8,>PZ016555.1 |Influenza A virus (A/Fox/WA/W2602...,ATGGAGAGAATAAAAGAGCTAAGAGATTTGATGTCGCAGTCTCGCA...
1,PZ016556.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8,>PZ016556.1 |Influenza A virus (A/Fox/WA/W2602...,ATGGATGTCAATCCGACTTTACTTTTCTTAAAAGTGCCAGCGCAAG...
2,PZ016557.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8,>PZ016557.1 |Influenza A virus (A/Fox/WA/W2602...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...
3,PZ016558.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8,>PZ016558.1 |Influenza A virus (A/Fox/WA/W2602...,ATGGAGAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...
4,PZ016559.1,GenBank,GCA_055411775.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Falghoush,A.M.","Washington state university, Veterinary Microb...",USA,NaN,2025-12-24,2026-02-18,ssRNA(-),8,>PZ016559.1 |Influenza A virus (A/Fox/WA/W2602...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAGATGGAGACTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124075,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205883.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...
124076,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205884.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...
124077,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205885.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...
124078,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205886.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...


## Find genotypes

(Using old genoflu results or Andersen Lab genoflu output) <br>
Old genoflu results should accumulate into one file to avoid having to genotype anything again.

### Find old genotypes

In [7]:
# Switch directory to previous week
os.chdir(prev_downloads_saved)

# Get both files from previous week
genoflu_output = pd.read_csv("output.tsv", delimiter="\t")
genoflu_results = pd.read_csv("results.tsv", delimiter="\t")

# Get old results from output.tsv
genoflu_old = pd.concat([genoflu_output, genoflu_results])
# genoflu_old = genoflu_old.rename(columns={"Strain":"Partial_Header"}) # So we can merge
print(genoflu_old)

# Switch back directory
os.chdir(downloads_saved)

# Save this output for the future
genoflu_old.to_csv("output.tsv", sep="\t", index=False)

                                               Strain  \
0   Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
1   Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
2   Influenza_A_virus__Mexico__Ciudad_de_Mexico_CP...   
3   Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
4   Influenza_A_virus__Mexico__Michoacan_CPA_02011...   
..                                                ...   
70                                    GCA_054697095_1   
71                                    GCA_054697065_1   
72                                    GCA_054697345_1   
73                                    GCA_054697675_1   
74                                    GCA_054697625_1   

                                             Genotype  \
0   Not assigned: Only 3 segments >98.0% match fou...   
1   Not assigned: Only 3 segments >98.0% match fou...   
2   Not assigned: Only 3 segments >98.0% match fou...   
3   Not assigned: Only 3 segments >98.0% match fou...   
4   Not assigned: Only 6 segme

In [8]:

metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate

for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)

# Merge to get already-genotyped segments
genoflu_old["Partial_Header_temp"] = genoflu_old["Strain"].apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1]) # Get only isolate 
genoflu_old["Partial_Header_temp"] = genoflu_old["Partial_Header_temp"].apply(lambda x: re.split(r'_H.N._20.{2}_.{2}_.{2}', x)[0]) # Get only isolate

metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Partial_Header_temp") 

# Isolate not-already-genotyped segments
metadata_segments_new = metadata_segments.merge(genoflu_old, indicator=True, how='left', on="Partial_Header_temp").loc[lambda x : x['_merge']=='left_only'] 

print(len(metadata_segments_old))
print(len(metadata_segments_new))

89336
35424


In [9]:
# Get genoflu results from Andersen and see if any fit

os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"}) # So we can merge with old
# genoflu_andersen["SRA_Accession"] = genoflu_andersen["Strain"] # So we can merge with new

# Find those genotyped by Andersen via merge
metadata_segments_known_andersen = metadata_segments_new.merge(genoflu_andersen, how="inner", on="SRA_Accession")
print(metadata_segments_known_andersen)
# Rename genotype by Andersen to concatenate
metadata_segments_known_andersen["Genotype_y"] = metadata_segments_known_andersen["Genotype"]

# Keep all known genotypes
metadata_segments_known = pd.concat([metadata_segments_old, metadata_segments_known_andersen]) # , on="SRA_Accession", how="left") # Since we know both of these

print(metadata_segments_old)
print(metadata_segments_known_andersen)

       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0     PZ008639.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
1     PZ008640.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
2     PZ008641.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
3     PZ008642.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
4     PZ008643.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
...          ...            ...              ...           ...           ...   
6171  PQ012131.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
6172  PQ012132.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
6173  PQ012133.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
6174  PQ012134.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
6175  PQ012135.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   

        BioProject      Organism_Name  

### Create FASTA files of unknown genotypes 

In [10]:
metadata_segments_known

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Genotype Average Depth of Coverage List_x,_merge,date,File Name,Genotype,"Genotype List Used, >=98.0%_y",Genotype Sample Title List_y,Genotype Percent Match List_y,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y
0,PX831491.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PX831492.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PX831493.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PX831494.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PX831495.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6171,PQ012131.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
6172,PQ012132.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
6173,PQ012133.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report
6174,PQ012134.1,GenBank,GCA_040780295.1,SRR29281472,SAMN41656639,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-50,SRR29281472.fa,B3.13,"NP:am8, NA:ea1, NS:am1.1, PB2:am2.2, MP:ea1, H...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","99.13%, 98.94%, 99.28%, 98.73%, 98.88%, 98.71%...","13, 15, 6, 29, 11, 22, 22, 10",Ran on FASTA - No Coverage Report


In [12]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_new["Partial_Header"] = metadata_segments_new["Assembly"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name

print(metadata_segments_new["Partial_Header"].values[0:5])

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_new["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_new[metadata_segments_new["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments_new["Partial_Header"])

print(df_list[0]["full_header"].values[:10])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

['>GCA_055411775.1' '>GCA_055411775.1' '>GCA_055411775.1'
 '>GCA_055411775.1' '>GCA_055411775.1']
0         >GCA_055411775.1
1         >GCA_055411775.1
2         >GCA_055411775.1
3         >GCA_055411775.1
4         >GCA_055411775.1
                ...       
124755    >GCA_038165665.1
124756    >GCA_038165665.1
124757    >GCA_038165665.1
124758    >GCA_038165665.1
124759    >GCA_038165665.1
Name: Partial_Header, Length: 35424, dtype: object
['>GCA_054871925_1' '>GCA_054871925_1' '>GCA_054871925_1'
 '>GCA_054871925_1' '>GCA_054871925_1' '>GCA_054871925_1'
 '>GCA_054871925_1' '>GCA_054871925_1']


### Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the NCBI_Virus downloads directory. <br>

To activate genoflu conda environment in BioWulf:
```
source myconda
conda activate genoflu
```

To run GenoFLU-multi, change directories to Multi-GenoFLU directory:
``` 
cd GenoFLU-multi
```

And then call the python script:

``` 
python bin/genoflu-multi.py -f <FASTA_directory>
```

In [ ]:
# Ensure that user does the above
input("Use genoflu. Afterwards, press ENTER to continue.")

''

In [13]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t") # This also shows the IDs of the new sequences -- incorporate into sequences report

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Assembly"] 
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])
# output_genoflu["Partial_Header_Merge"] = output_genoflu["Partial_Header_Merge"].apply(lambda x: re.split(r'_H.N._(20|19).{2}_.{2}_.{2}', x)[0])

# Merge
metadata_genoflu = metadata_segments_new.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill(limit_area="inside")



print(metadata_genoflu)


       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0     PZ008639.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
1     PZ008640.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
2     PZ008641.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
3     PZ008642.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
4     PZ008643.1        GenBank  GCA_055411335.1   SRR36856000  SAMN54681016   
...          ...            ...              ...           ...           ...   
1243  PX934299.1        GenBank  GCA_054864435.1           NaN           NaN   
1244  PX934300.1        GenBank  GCA_054864435.1           NaN           NaN   
1245  PX934301.1        GenBank  GCA_054864435.1           NaN           NaN   
1246  PX934302.1        GenBank  GCA_054864435.1           NaN           NaN   
1247  PX934303.1        GenBank  GCA_054864435.1           NaN           NaN   

        BioProject      Organism_Name  

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_58640\2836936310.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill(limit_area="inside")


### Concatenate with known genotypes

In [14]:
# Rename columns so we can concatenate
print(metadata_segments_known.columns)

metadata_segments_known = metadata_segments_known.reset_index(drop=True)
print(metadata_segments_known)

print(metadata_genoflu.columns)
metadata_segments_known["Genotype_official"] = metadata_segments_known["Genotype_y"]
metadata_segments_known["Serotype"] = metadata_segments_known["Genotype_x"]
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype"]
metadata_genoflu["Serotype"] = metadata_genoflu["Genotype_x"]

metadata_genoflu = metadata_genoflu.reset_index(drop=True)
print(metadata_genoflu)

# Concatenation
metadata_genoflu_concat = pd.concat([metadata_segments_known, metadata_genoflu])

metadata_genoflu_concat

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family',
       'Genotype_x', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'size', 'full_header', 'sequence', 'Partial_Header_temp', 'Strain',
       'Genotype_y', 'Genotype List Used, >=98.0%',
       'Genotype Sample Title List', 'Genotype Percent Match List',
       'Genotype Mismatch List', 'Genotype Average Depth of Coverage List',
       'Date run', 'Genotype List Used, >=98.0%_x',
       'Genotype Sample Title List_x', 'Genotype Percent Match List_x',
       'Genotype Mismatch List_x', 'Genotype Average Depth of Coverage List_x',
       '_merge', 'date', 'File Name', 'Genotype',
       'Genotype List Used, >=98.0

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y,Genotype_official,Serotype,Strain_x,Date run_x,Partial_Header,Partial_Header_Merge,Strain_y,Date run_y
0,PX831491.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,B3.13,H5N1,NaN,NaN,NaN,NaN,NaN,NaN
1,PX831492.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,B3.13,H5N1,NaN,NaN,NaN,NaN,NaN,NaN
2,PX831493.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,B3.13,H5N1,NaN,NaN,NaN,NaN,NaN,NaN
3,PX831494.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,B3.13,H5N1,NaN,NaN,NaN,NaN,NaN,NaN
4,PX831495.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,B3.13,H5N1,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,PX934299.1,GenBank,GCA_054864435.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"8, 8, 16, 4, 9, 1, 10, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_054864435.1,GCA_054864435_1,GCA_054864435_1,2026-02-24_09-45-23
1244,PX934300.1,GenBank,GCA_054864435.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"8, 8, 16, 4, 9, 1, 10, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_054864435.1,GCA_054864435_1,GCA_054864435_1,2026-02-24_09-45-23
1245,PX934301.1,GenBank,GCA_054864435.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"8, 8, 16, 4, 9, 1, 10, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_054864435.1,GCA_054864435_1,GCA_054864435_1,2026-02-24_09-45-23
1246,PX934302.1,GenBank,GCA_054864435.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"8, 8, 16, 4, 9, 1, 10, 7",Ran on FASTA - No Coverage Report,D1.1,H5N1,NaN,NaN,>GCA_054864435.1,GCA_054864435_1,GCA_054864435_1,2026-02-24_09-45-23


In [15]:
# Cut down to only columns we want
metadata_genoflu_concat = metadata_genoflu_concat[["Accession", "Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype_official", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header"]] #, "Strain"]]

# Get genbank strain name
metadata_genoflu_concat["genbank_name"] = metadata_genoflu_concat["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
metadata_genoflu_concat["Host"] = metadata_genoflu_concat["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

metadata_genoflu_concat = metadata_genoflu_concat.dropna(subset="genbank_name") # [metadata_genoflu["Genotype"]  == "B3.13"]

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_58640\4267596827.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_genoflu_concat["genbank_name"] = metadata_genoflu_concat["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_58640\4267596827.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_genoflu_concat["Host"] = metadata_genoflu_concat["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

In [16]:
metadata_genoflu_concat

# Make sure we only have the serotype(s) we want
for serotype in serotypes:
    metadata_genoflu = metadata_genoflu[metadata_genoflu["Serotype"] == serotype]

## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [17]:
# Re-re-name so that we don't break following code

metadata_genoflu = metadata_genoflu_concat

After running the below code, **STOP TO CHECK** if any new animals appear

In [19]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['rock goose', 'antofagasta', 'common loon', 'crow', 'common grackle', 'northern shoveler', 'tundra swan', 'double-crested cormorant', 'pekin duck', 'green heron', 'domestic grower', 'black-legged kittiwake', 'dolphin', 'brant', 'gannet', 'red-shouldered hawk', 'rough-legged hawk', 'pheasant', 'numida meleagris', 'pet food', 'pelican', 'vulture', 'sparrow', 'short-billed gull', 'whooping crane', 'owl', 'tiger', 'long-eared owl', 'black-crowned night-heron', 'serval', 'muscovy duck', 'virginia opossum', 'feline', 'ring-necked duck', 'loon, common', 'western sandpiper', 'green winged teal', 'red-necked phalarope', 'gull', 'royal tern', 'emu', 'poultry', 'goat', 'barred owl', 'heron', 'american wood stork', 'waterfowl', 'lesser snow goose blue-morph', 'bird', 'domestic goose', 'raccoon', 'raw pet food', 'grackle', 'ring-necked pheasant', 'hermit thrush', 'buteogallus urubitinga', 'chukar partridge', 'peafowl', 'bufflehead duck', 'trumpeter swan', 'wild duck', 'great black-backed gull', 'a

In [ ]:
input("Check animals output. Afterwards, press ENTER to continue.")

''

In [ ]:
# Re-label sequences with no assigned genotype as "Unassigned"

metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype_official"].apply(lambda x: "Not assigned" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"
metadata_genoflu["Genotype"] = metadata_genoflu["Genotype_official"] # Rename column again to not break old code

In [22]:
metadata_genoflu

,Accession,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,genbank_name,Genotype
0,PX831491.1,GCA_054448965.1,Influenza A virus (A/Dairy cow/Texas/063224-24...,Dairy cow,2024-03,NaN,063224-24-1,B3.13,USA: Texas,>PX831491.1 |Influenza A virus (A/Dairy cow/Te...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAAC...,H5N1,1,NaN,NaN,A/Dairy cow/Texas/063224-24-1/2024,B3.13
1,PX831492.1,GCA_054448965.1,Influenza A virus (A/Dairy cow/Texas/063224-24...,Dairy cow,2024-03,NaN,063224-24-1,B3.13,USA: Texas,>PX831492.1 |Influenza A virus (A/Dairy cow/Te...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACCTTAC...,H5N1,2,NaN,NaN,A/Dairy cow/Texas/063224-24-1/2024,B3.13
2,PX831493.1,GCA_054448965.1,Influenza A virus (A/Dairy cow/Texas/063224-24...,Dairy cow,2024-03,NaN,063224-24-1,B3.13,USA: Texas,>PX831493.1 |Influenza A virus (A/Dairy cow/Te...,AGCAAAAGCAGGTACTGATTCAAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,NaN,NaN,A/Dairy cow/Texas/063224-24-1/2024,B3.13
3,PX831494.1,GCA_054448965.1,Influenza A virus (A/Dairy cow/Texas/063224-24...,Dairy cow,2024-03,NaN,063224-24-1,B3.13,USA: Texas,>PX831494.1 |Influenza A virus (A/Dairy cow/Te...,AGCAAAAGCAGGGGTTCACTCTGTCAAAATGGAGAACATAGTACTA...,H5N1,4,NaN,NaN,A/Dairy cow/Texas/063224-24-1/2024,B3.13
4,PX831495.1,GCA_054448965.1,Influenza A virus (A/Dairy cow/Texas/063224-24...,Dairy cow,2024-03,NaN,063224-24-1,B3.13,USA: Texas,>PX831495.1 |Influenza A virus (A/Dairy cow/Te...,AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCA...,H5N1,5,NaN,NaN,A/Dairy cow/Texas/063224-24-1/2024,B3.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,PX934299.1,GCA_054864435.1,Influenza A virus (A/Dairy Cow/NV/W250550062-1...,Dairy Cow,2025-02-23,NaN,W250550062-1,D1.1,USA: NV,>PX934299.1 |Influenza A virus (A/Dairy Cow/NV...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,H5N1,4,NaN,>GCA_054864435.1,A/Dairy Cow/NV/W250550062-1/2025,D1.1
1244,PX934300.1,GCA_054864435.1,Influenza A virus (A/Dairy Cow/NV/W250550062-1...,Dairy Cow,2025-02-23,NaN,W250550062-1,D1.1,USA: NV,>PX934300.1 |Influenza A virus (A/Dairy Cow/NV...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,NaN,>GCA_054864435.1,A/Dairy Cow/NV/W250550062-1/2025,D1.1
1245,PX934301.1,GCA_054864435.1,Influenza A virus (A/Dairy Cow/NV/W250550062-1...,Dairy Cow,2025-02-23,NaN,W250550062-1,D1.1,USA: NV,>PX934301.1 |Influenza A virus (A/Dairy Cow/NV...,ATGAATCCAAATCAAAAGATAATAACTATCGGGTCAATCTGCATGG...,H5N1,6,NaN,>GCA_054864435.1,A/Dairy Cow/NV/W250550062-1/2025,D1.1
1246,PX934302.1,GCA_054864435.1,Influenza A virus (A/Dairy Cow/NV/W250550062-1...,Dairy Cow,2025-02-23,NaN,W250550062-1,D1.1,USA: NV,>PX934302.1 |Influenza A virus (A/Dairy Cow/NV...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,NaN,>GCA_054864435.1,A/Dairy Cow/NV/W250550062-1/2025,D1.1


In [23]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# Get geographic locations
metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        x
                                                                        )

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


print(metadata_genoflu["Geo_Location_Abrv"])

0       USA-TX
1       USA-TX
2       USA-TX
3       USA-TX
4       USA-TX
         ...  
1243    USA-NV
1244    USA-NV
1245    USA-NV
1246    USA-NV
1247    USA-NV
Name: Geo_Location_Abrv, Length: 96760, dtype: object


In [24]:
# If there is no SRA Accession, replace identifier with Accession
metadata_genoflu["SRA_Accession"] = np.where(metadata_genoflu['SRA_Accession'] == "", metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]), metadata_genoflu['SRA_Accession'])
# If there is no Assembly, replace identifier with Accession -- only for PB2
metadata_genoflu["Identifier"] = np.where(metadata_genoflu["SRA_Accession"] == "", metadata_genoflu["Accession"].apply(lambda x: metadata_genoflu[metadata_genoflu["Accession"] == x] if metadata_genoflu[metadata_genoflu["Accession"] == x].loc[:, "Segment"].values[0] == 1 else np.nan), metadata_genoflu["SRA_Accession"])
# Fill in other nans with PB2 Accession (interpolate, maximum of 7 other sequences)
metadata_genoflu.loc[:,"Identifier"] = metadata_genoflu.loc[:, "Identifier"].ffill(limit=7, limit_area="inside")

print(metadata_genoflu)

       Accession         Assembly  \
0     PX831491.1  GCA_054448965.1   
1     PX831492.1  GCA_054448965.1   
2     PX831493.1  GCA_054448965.1   
3     PX831494.1  GCA_054448965.1   
4     PX831495.1  GCA_054448965.1   
...          ...              ...   
1243  PX934299.1  GCA_054864435.1   
1244  PX934300.1  GCA_054864435.1   
1245  PX934301.1  GCA_054864435.1   
1246  PX934302.1  GCA_054864435.1   
1247  PX934303.1  GCA_054864435.1   

                                          GenBank_Title       Host  \
0     Influenza A virus (A/Dairy cow/Texas/063224-24...  dairy cow   
1     Influenza A virus (A/Dairy cow/Texas/063224-24...  dairy cow   
2     Influenza A virus (A/Dairy cow/Texas/063224-24...  dairy cow   
3     Influenza A virus (A/Dairy cow/Texas/063224-24...  dairy cow   
4     Influenza A virus (A/Dairy cow/Texas/063224-24...  dairy cow   
...                                                 ...        ...   
1243  Influenza A virus (A/Dairy Cow/NV/W250550062-1...  dairy co

In [25]:

# Make new labels
names = ">" + metadata_genoflu["Identifier"].astype(str) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"].astype(str) + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names



In [26]:
os.chdir(complete_files)

# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

# Save metadata
metadata_genoflu.to_csv("NCBI_Virus_" + date_range + "_metadata.csv") # Make file for metadata


96760
95216


## Rename segments and make complete FASTA files

In [ ]:
# Set up segments

if len(genotypes) > 3: # If we're not doing maintenance only
    genotypes.append("Not assigned") # Make sure unassigned genotypes are included
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

B3.13_PB2
D1.3_PB2
D1.1_PB2
B3.13_PB1
D1.3_PB1
D1.1_PB1
B3.13_PA
D1.3_PA
D1.1_PA
B3.13_HA
D1.3_HA
D1.1_HA
B3.13_NP
D1.3_NP
D1.1_NP
B3.13_NA
D1.3_NA
D1.1_NA
B3.13_MP
D1.3_MP
D1.1_MP
B3.13_NS
D1.3_NS
D1.1_NS


In [28]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

       Accession         Assembly  \
0     PX831491.1  GCA_054448965.1   
120   PX806572.1  GCA_054433815.1   
168   PX806620.1  GCA_054435515.1   
2056  PX809220.1  GCA_054429805.1   
2064  PX809228.1  GCA_054434675.1   
...          ...              ...   
911   PX936693.1  GCA_054872375.1   
919   PX936709.1  GCA_054872415.1   
927   PX936717.1  GCA_054872425.1   
935   PX936725.1  GCA_054872495.1   
1071  PX936981.1  GCA_054872785.1   

                                          GenBank_Title       Host  \
0     Influenza A virus (A/Dairy cow/Texas/063224-24...  dairy cow   
120   Influenza A virus (A/cattle/CA/25-026029-002-o...     cattle   
168   Influenza A virus (A/cattle/CA/25G02121-002-or...     cattle   
2056  Influenza A virus (A/cattle/CA/25-024071-005-o...     cattle   
2064  Influenza A virus (A/cattle/CA/25-026426-003-o...     cattle   
...                                                 ...        ...   
911   Influenza A virus (A/Bovine/California/B240054...     bovin